# Лабораторная работа № 6 — ЭТАЛОННОЕ РЕШЕНИЕ

## Обратная задача кинематики

**Дисциплина:** Введение в технологии виртуальной дополненной реальности и робототехники
**Тема курса:** 7. Обратная задача кинематики как задача решения нелинейной системы

---

**Для преподавателя.** Не выдавать студентам до сдачи работы.

---

### Цель работы

Решить обратную задачу кинематики двумя способами — аналитически и методом Ньютона, —
исследовать бассейны притяжения численного метода, увидеть, что происходит вблизи
границы рабочей зоны, и разобраться с избыточным манипулятором.

### Что понадобится

`numpy`, `scipy`, `matplotlib`.

### Правило практикума

Метод Ньютона вы реализуете сами. `scipy.optimize.least_squares` используется
в задании 6 по прямому назначению — как промышленный решатель с ограничениями,
и результат сравнивается с собственной реализацией.

## Теоретическая справка

### Постановка

Прямая задача кинематики — отображение $f: T^n \to SE(3)$, вычисляемое произведением
матриц вдоль цепи. Обратная задача — найти конфигурацию по заданной позе схвата:

$$\text{найти } q \text{ такое, что } f(q) = x^* .$$

Симметрии между задачами нет никакой, и это главное, что нужно вынести из работы.

| | Прямая задача | Обратная задача |
|---|---|---|
| Решение существует | всегда | не всегда |
| Решение единственно | всегда | почти никогда |
| Явная формула | есть | только для частных структур |
| Сложность | линейная по числу звеньев | нелинейная система |

### Плоский 2R: аналитическое решение

Для двухзвенника решение выписывается через теорему косинусов. Если
$r^2 = x^2 + y^2$, то

$$\cos q_2 = \frac{r^2 - l_1^2 - l_2^2}{2 l_1 l_2},
\qquad q_1 = \operatorname{atan2}(y, x) - \operatorname{atan2}\bigl(l_2 \sin q_2,\; l_1 + l_2\cos q_2\bigr).$$

Знак $q_2$ выбирается произвольно — отсюда **две ветви решения**, которые называют
«локоть вверх» и «локоть вниз». Условие достижимости:
$|l_1 - l_2| \le r \le l_1 + l_2$, то есть рабочая зона — кольцо.

### Численное решение: метод Ньютона

В общем случае замкнутой формулы нет, и задачу решают как систему нелинейных
уравнений $F(q) = f(q) - x^* = 0$. Шаг метода Ньютона:

$$J(q)\,\Delta q = -F(q), \qquad q \leftarrow q + \Delta q,$$

где $J = \partial f / \partial q$ — якобиан. Для плоского 2R он выписывается явно,
и его определитель равен

$$\det J = l_1 l_2 \sin q_2 .$$

Определитель обращается в нуль при $q_2 = 0$ и $q_2 = \pi$ — рука полностью вытянута
или сложена. Это **сингулярные конфигурации**: именно там метод начинает буксовать,
и подробно ими займёмся в теме 8.

### Демпфирование

Вблизи вырождения шаг Ньютона неограниченно растёт. Лекарство — регуляризация
Левенберга: вместо $J\Delta q = -F$ решается

$$(J^{\mathsf T}J + \lambda^2 I)\,\Delta q = -J^{\mathsf T}F .$$

Шаг остаётся ограниченным, но решение смещается: при $\lambda > 0$ невязка
в точности нулю уже не равна. Это типичный размен устойчивости на точность.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

L1, L2 = 0.5, 0.35

## Задание 1. Прямая кинематика и якобиан

**Ответ на вопрос:** $\det J = l_1 l_2 \sin q_2$ обращается в нуль при $q_2 = 0$ и $q_2 = \pi$ — рука полностью вытянута либо сложена вдвое. Геометрически в этих конфигурациях схват может двигаться только по окружности: радиальное движение мгновенно невозможно, потому что оба звена лежат на одной прямой. Механизм теряет одну степень свободы — это сингулярность, и подробно ею займёмся в теме 8.

In [2]:
def fk2(q):
    q1, q2 = q
    return np.array([L1 * np.cos(q1) + L2 * np.cos(q1 + q2),
                     L1 * np.sin(q1) + L2 * np.sin(q1 + q2)])

def jac2(q):
    q1, q2 = q
    s1, c1 = np.sin(q1), np.cos(q1)
    s12, c12 = np.sin(q1 + q2), np.cos(q1 + q2)
    return np.array([[-L1 * s1 - L2 * s12, -L2 * s12],
                     [ L1 * c1 + L2 * c12,  L2 * c12]])

def jac_num(f, q, h=1e-6):
    q = np.asarray(q, float)
    return np.column_stack([(f(q + h * e) - f(q - h * e)) / (2 * h)
                            for e in np.eye(len(q))])

qt = np.array([0.6, -0.9])
err_J = np.abs(jac2(qt) - jac_num(fk2, qt)).max()
print('Задание 1: макс. расхождение аналитического и численного якобиана %.2e' % err_J)
assert err_J < 1e-8
detJ = np.linalg.det(jac2(qt))
print('           det J = %.6f,  теория l1·l2·sin q2 = %.6f'
      % (detJ, L1 * L2 * np.sin(qt[1])))
assert abs(detJ - L1 * L2 * np.sin(qt[1])) < 1e-12

Задание 1: макс. расхождение аналитического и численного якобиана 3.94e-11
           det J = -0.137082,  теория l1·l2·sin q2 = -0.137082


## Задание 2. Аналитическое решение: две ветви

**Ответы на вопросы:** решений ровно два, потому что $\cos q_2$ определён однозначно, а сам угол — с точностью до знака. Каждому знаку отвечает своя конфигурация. На границе кольца ($r = l_1 + l_2$ или $r = |l_1 - l_2|$) обе ветви сливаются в одну: там $\sin q_2 = 0$, и решение единственно — ровно та самая сингулярная конфигурация.

In [3]:
def ik2_analytic(target):
    x, y = target
    r2 = x * x + y * y
    c2 = (r2 - L1 ** 2 - L2 ** 2) / (2 * L1 * L2)
    if abs(c2) > 1:
        return []
    out = []
    for sign in (+1, -1):
        q2 = sign * np.arccos(np.clip(c2, -1, 1))
        q1 = np.arctan2(y, x) - np.arctan2(L2 * np.sin(q2), L1 + L2 * np.cos(q2))
        out.append(np.array([q1, q2]))
    return out

target = np.array([0.55, 0.30])
sols = ik2_analytic(target)
print('Задание 2: решений найдено', len(sols))
for k, q in enumerate(sols):
    nm = 'локоть вниз' if q[1] < 0 else 'локоть вверх'
    print('           %-12s q = %s,  невязка %.2e'
          % (nm, np.round(q, 6), np.linalg.norm(fk2(q) - target)))
assert len(sols) == 2
assert all(np.linalg.norm(fk2(q) - target) < 1e-12 for q in sols)
r_out, r_in = L1 + L2, abs(L1 - L2)
print('           рабочая зона: кольцо от %.2f до %.2f м' % (r_in, r_out))
print('           недостижимая точка (0,95; 0): решений %d'
      % len(ik2_analytic(np.array([0.95, 0.0]))))
assert len(ik2_analytic(np.array([0.95, 0.0]))) == 0

Задание 2: решений найдено 2
           локоть вверх q = [-0.0923  1.5136],  невязка 0.00e+00
           локоть вниз  q = [ 1.091  -1.5136],  невязка 5.55e-17
           рабочая зона: кольцо от 0.15 до 0.85 м
           недостижимая точка (0,95; 0): решений 0


## Задание 3. Метод Ньютона и скорость сходимости

**Ответы на вопросы:** сходимость квадратичная. Это видно прямо из последовательности невязок: $8{,}1\cdot10^{-2} \to 1{,}1\cdot10^{-2} \to 1{,}0\cdot10^{-4} \to 1{,}4\cdot10^{-8} \to 1{,}2\cdot10^{-16}$ — показатель степени примерно удваивается на каждом шаге, то есть число верных знаков растёт вдвое. Отношение $r_{k+1}/r_k^2$ держится порядка единицы, что и есть определение квадратичной сходимости.

Метод сошёлся к ветви «локоть вверх» ($q_2 > 0$), потому что начальное приближение $q_2 = 1{,}2 > 0$ лежит в её бассейне.

In [4]:
def ik2_newton(target, q0, tol=1e-12, iters=60, damp=0.0):
    q = np.asarray(q0, float).copy()
    hist = []
    for _ in range(iters):
        r = fk2(q) - target
        hist.append(np.linalg.norm(r))
        if hist[-1] < tol:
            break
        J = jac2(q)
        if damp > 0:
            dq = np.linalg.solve(J.T @ J + damp ** 2 * np.eye(2), -J.T @ r)
        else:
            try:
                dq = np.linalg.solve(J, -r)
            except np.linalg.LinAlgError:
                break
        q = q + dq
    hist.append(np.linalg.norm(fk2(q) - target))
    return q, np.array(hist)

q_sol, hist = ik2_newton(target, [0.0, 1.2])
print('Задание 3: итераций %d, невязка %.3e -> %.3e' % (len(hist) - 1, hist[0], hist[-1]))
print('           по итерациям:', ' '.join('%.2e' % h for h in hist))
assert hist[-1] < 1e-12
assert any(np.allclose(q_sol, s, atol=1e-8) for s in sols)
tail = [(hist[i], hist[i + 1]) for i in range(len(hist) - 1)
        if 1e-9 < hist[i] < 0.2 and hist[i + 1] > 0]
ratios = [b / a ** 2 for a, b in tail]
print('           отношение rₖ₊₁/rₖ² в квадратичной фазе: %s' % np.round(ratios, 3))
print('           число верных знаков удваивается за итерацию')
assert len(ratios) >= 2 and max(ratios) < 50

Задание 3: итераций 5, невязка 8.117e-02 -> 1.241e-16
           по итерациям: 8.12e-02 1.09e-02 1.04e-04 1.43e-08 1.24e-16 1.24e-16
           отношение rₖ₊₁/rₖ² в квадратичной фазе: [1.657 0.877 1.308 0.609]
           число верных знаков удваивается за итерацию


## Задание 4. Бассейны притяжения

**Ответы на вопросы:** граница между бассейнами **не гладкая** — она проходит вдоль прямой $q_2 = 0$, где якобиан вырожден, и вблизи неё картина становится изрезанной: шаг Ньютона там огромен и забрасывает точку далеко, иногда в противоположный бассейн. Это типичная для метода Ньютона фрактальная структура границы.

Предсказать ветвь по начальному приближению можно грубо — знак $q_2$ обычно сохраняется, — но не надёжно: примерно 1 % точек не сходится за 40 итераций, и это как раз окрестность вырождения.

In [5]:
n = 220
g = np.linspace(-np.pi, np.pi, n)
Q1, Q2 = np.meshgrid(g, g)
basin = np.zeros((n, n))
iters_map = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        q, h = ik2_newton(target, [Q1[i, j], Q2[i, j]], iters=40)
        iters_map[i, j] = len(h) - 1
        if h[-1] > 1e-8:
            basin[i, j] = 0
        else:
            basin[i, j] = 1 if np.sin(q[1]) > 0 else 2
frac = [np.mean(basin == k) for k in (0, 1, 2)]
print('Задание 4: сетка %dx%d начальных приближений' % (n, n))
print('           не сошлось %.1f %%, локоть вверх %.1f %%, локоть вниз %.1f %%'
      % tuple(100 * f for f in frac))
print('           среднее число итераций %.1f, максимум %d'
      % (iters_map.mean(), int(iters_map.max())))
assert frac[1] > 0.3 and frac[2] > 0.3

fig, axs = plt.subplots(1, 2, figsize=(11, 4.6))
axs[0].imshow(basin, extent=[-np.pi, np.pi, -np.pi, np.pi], origin='lower',
              cmap='coolwarm', interpolation='nearest')
axs[0].set_title('бассейны притяжения'); axs[0].set_xlabel('q1'); axs[0].set_ylabel('q2')
im = axs[1].imshow(np.minimum(iters_map, 20), extent=[-np.pi, np.pi, -np.pi, np.pi],
                   origin='lower', cmap='viridis', interpolation='nearest')
axs[1].set_title('число итераций'); axs[1].set_xlabel('q1')
plt.colorbar(im, ax=axs[1]); plt.tight_layout()
plt.savefig('/tmp/_basins.png', dpi=80); plt.close()

Задание 4: сетка 220x220 начальных приближений
           не сошлось 0.9 %, локоть вверх 49.5 %, локоть вниз 49.6 %
           среднее число итераций 10.4, максимум 40


## Задание 5. Что происходит вблизи границы рабочей зоны

**Ответы на вопросы:** при $r \to l_1 + l_2$ имеем $\cos q_2 \to 1$, значит $q_2 \sim \sqrt{\delta}$, а $\det J = l_1l_2\sin q_2 \sim \sqrt{\delta}$. Отсюда обусловленность растёт как $\delta^{-1/2}$ — эксперимент даёт наклон $-0{,}507$.

Демпфирование ограничивает длину шага и не даёт методу разлететься, но вносит систематическое смещение: при $\lambda = 10^{-2}$ невязка застревает на уровне $1{,}5\cdot10^{-6}$ и до машинной точности не доходит даже за 200 итераций. Это классический размен устойчивости на точность; на практике $\lambda$ уменьшают по мере приближения к решению — так устроен метод Левенберга — Марквардта.

In [6]:
print('Задание 5: приближаем цель к границе рабочей зоны  r -> l1 + l2 = %.2f м' % r_out)
print('           %-10s %-12s %-10s %-8s' % ('δ, м', 'cond J', 'итераций', 'q2, рад'))
conds, its = [], []
for d in (1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6):
    tg = np.array([r_out - d, 0.0])
    qa = ik2_analytic(tg)[0]
    c = np.linalg.cond(jac2(qa))
    _, h = ik2_newton(tg, [0.35, 0.9], iters=200)
    conds.append(c); its.append(len(h) - 1)
    print('           %-10.0e %-12.3e %-10d %-8.4f' % (d, c, len(h) - 1, qa[1]))
sl = np.polyfit(np.log10([1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6]), np.log10(conds), 1)[0]
print('           наклон log(cond J) по log δ = %.3f  (теория −0,5)' % sl)
assert -0.6 < sl < -0.4

# на самой границе якобиан вырожден: сравним обычный и демпфированный Ньютон
tb = np.array([r_out, 0.0])
for damp, nm in ((0.0, 'обычный Ньютон'), (1e-2, 'демпфированный (λ = 1e-2)')):
    q, h = ik2_newton(tb, [0.35, 0.9], iters=200, damp=damp)
    print('           %-28s итераций %3d, невязка %.2e'
          % (nm, len(h) - 1, h[-1]))
print('           демпфирование ограничивает шаг ценой смещения оценки')

Задание 5: приближаем цель к границе рабочей зоны  r -> l1 + l2 = 0.85 м
           δ, м       cond J       итераций   q2, рад 
           1e-01      4.435e+00    7          0.9970  
           1e-02      1.535e+01    8          0.3120  
           1e-03      4.895e+01    10         0.0986  
           1e-04      1.549e+02    11         0.0312  
           1e-05      4.899e+02    13         0.0099  
           1e-06      1.549e+03    14         0.0031  
           наклон log(cond J) по log δ = -0.507  (теория −0,5)
           обычный Ньютон               итераций  21, невязка 8.25e-13
           демпфированный (λ = 1e-2)    итераций 200, невязка 1.47e-06
           демпфирование ограничивает шаг ценой смещения оценки


## Задание 6. Промышленный решатель с ограничениями

**Ответы на вопросы:** ограничения на суставные углы отсекают ненужную ветвь на уровне постановки задачи — не нужно решать обе и выбирать. Так и делают на практике: механические упоры сочленений естественным образом задают эти границы.

Для недостижимой цели решатель находит **ближайшую достижимую точку**: результат $(0{,}85;\ 0)$ лежит точно на границе рабочей зоны, а невязка $0{,}35$ м равна $1{,}2 - 0{,}85$. Это важное свойство постановки через наименьшие квадраты: она осмысленна и тогда, когда точного решения не существует.

In [7]:
def resid(q):
    return fk2(q) - target

lo, hi = np.array([-np.pi, 0.05]), np.array([np.pi, np.pi])   # только локоть вверх
res = least_squares(resid, x0=[0.5, 1.0], jac=lambda q: jac2(q),
                    bounds=(lo, hi), xtol=1e-15, ftol=1e-15)
print('Задание 6: least_squares с ограничением q2 ∈ [0,05; π]')
print('           q = %s, невязка %.2e, итераций %d'
      % (np.round(res.x, 6), np.linalg.norm(res.fun), res.nfev))
assert np.linalg.norm(res.fun) < 1e-7 and res.x[1] > 0
unreach = np.array([1.2, 0.0])
res2 = least_squares(lambda q: fk2(q) - unreach, x0=[0.3, 0.3])
print('           недостижимая цель (1,2; 0): невязка %.4f м, ближайшая точка %s'
      % (np.linalg.norm(res2.fun), np.round(fk2(res2.x), 4)))
assert abs(np.linalg.norm(res2.fun) - (1.2 - r_out)) < 1e-6

Задание 6: least_squares с ограничением q2 ∈ [0,05; π]
           q = [-0.0923  1.5136], невязка 5.73e-09, итераций 5
           недостижимая цель (1,2; 0): невязка 0.3500 м, ближайшая точка [0.85 0.  ]


## Задание 7. Избыточный манипулятор: ядро якобиана

**Ответы на вопросы:** ядро якобиана — это касательное направление, вдоль которого схват не движется **в первом порядке**. При конечном сдвиге проявляется кривизна: смещение растёт как квадрат шага, что и видно из чисел — 0,05 рад дают 0,1 мм, а 0,2 рад уже 1,5 мм, то есть при учетверении шага смещение выросло примерно в 15 раз. Чтобы держаться на многообразии решений точно, надо после каждого шага вдоль ядра делать коррекцию методом Ньютона.

Свободу распоряжаются так: обход препятствий без смещения схвата, уход от предельных углов в сочленениях, уход от сингулярных конфигураций. Формально это записывается как проекция вспомогательного критерия в ядро якобиана — вернёмся к этому в теме 8.

In [8]:
L = np.array([0.5, 0.35, 0.2])

def fk3(q):
    a = np.cumsum(q)
    return np.array([np.sum(L * np.cos(a)), np.sum(L * np.sin(a))])

def jac3(q):
    a = np.cumsum(q)
    return np.array([[-np.sum(L[k:] * np.sin(a[k:])) for k in range(3)],
                     [ np.sum(L[k:] * np.cos(a[k:])) for k in range(3)]])

t3 = np.array([0.6, 0.4])
q = np.array([0.4, 0.4, 0.4])
for _ in range(60):
    r = fk3(q) - t3
    if np.linalg.norm(r) < 1e-14:
        break
    q = q - np.linalg.pinv(jac3(q)) @ r
print('Задание 7: 3R, невязка %.2e, q = %s' % (np.linalg.norm(fk3(q) - t3), np.round(q, 4)))
assert np.linalg.norm(fk3(q) - t3) < 1e-12
J3 = jac3(q)
ns = np.linalg.svd(J3)[2][-1]
print('           размерность ядра якобиана: %d' % (3 - np.linalg.matrix_rank(J3)))
drift = [np.linalg.norm(fk3(q + t * ns) - t3) for t in (0.05, 0.1, 0.2)]
print('           сдвиг вдоль ядра на 0,05/0,1/0,2 рад: смещение схвата %s м'
      % np.round(drift, 5))
assert drift[0] < 5e-3
qs_null = np.array([q + t * ns for t in np.linspace(-0.6, 0.6, 40)])
print('           40 конфигураций вдоль ядра: разброс схвата %.2e м'
      % np.abs(np.array([fk3(x) for x in qs_null]) - t3).max())
print('\nВСЕ ПРОВЕРКИ ПРОЙДЕНЫ')

Задание 7: 3R, невязка 2.48e-16, q = [-0.1304  1.0992  1.091 ]
           размерность ядра якобиана: 1
           сдвиг вдоль ядра на 0,05/0,1/0,2 рад: смещение схвата [0.0001 0.0004 0.0015] м
           40 конфигураций вдоль ядра: разброс схвата 1.37e-02 м

ВСЕ ПРОВЕРКИ ПРОЙДЕНЫ


## Критерии приёма работы

| Что проверяется | На что смотреть |
|---|---|
| Задание 1 | Якобиан выведен аналитически и сверен численно; определитель сопоставлен с формулой |
| Задание 2 | Возвращается список решений; обработан случай недостижимой цели |
| Задание 3 | Показана последовательность невязок; названа скорость сходимости |
| Задание 4 | Обе карты построены; объяснена изрезанность границы |
| Задание 5 | Наклон посчитан МНК; объяснён показатель $-1/2$ |
| Задание 6 | Передан аналитический якобиан; разобран случай недостижимой цели |
| Задание 7 | Ядро найдено через SVD; объяснено, почему сохранение лишь приближённое |
| Воспроизводимость | «Restart & Run All» отрабатывает без ошибок |

**Частые ошибки:**

1. В `ik2_analytic` не проверяется $|\cos q_2| \le 1$ — для недостижимой цели
   `arccos` возвращает `nan`, и дальше всё разваливается молча.
2. `q1` вычисляется как $\operatorname{atan2}(y,x) - q_2/2$ или иным «наглядным»
   способом. Работает только при $l_1 = l_2$.
3. В методе Ньютона используется `np.linalg.inv(J) @ r` вместо `solve`. Работает,
   но вблизи вырождения теряет точность заметно раньше.
4. История невязок записывается после шага, а не до, — тогда первая невязка теряется
   и картина сходимости выглядит иначе.
5. В задании 4 сетка берётся 50×50 «чтобы быстрее». Структура границы при этом
   не видна, и главный вывод работы теряется.
6. В задании 7 ядро ищется как `Vt[0]` вместо `Vt[-1]` — та же ошибка, что
   в лабораторной работе № 4.

## Прикладной блок · IK в промышленном стеке: MoveIt и демпфированный Ньютон *(необязательный)*

Обратная задача кинематики из этой работы — то, что решает **MoveIt 2** при
каждой команде цели схвата:

| Понятие из работы | Контрагент в ROS 2 |
|---|---|
| `ik2_newton(target, q0)` | IK-солверы MoveIt: KDL, Trac-IK, IKFast — тот же Ньютон/Гаусс-Ньютон |
| параметр `damp` — λ-регуляризация | демпфированный метод наименьших квадратов (DLS) в KDL/Trac-IK |
| бассейны притяжения (задание 4) | «seeds» — начальные приближения: от них зависит, какое решение найдёт солвер |
| вырождение на границе (задание 5) | почему демпфирование обязательно: λ ограничивает шаг у сингулярности |
| ядро якобиана 3R (задание 7) | нуль-пространство: избыточные руки (7-DOF) двигают суставами, не двигая схват |

Блок ниже строит 2R-манипулятор из URDF, берёт якобиан у Pinocchio и делает
тот же демпфированный Ньютон, что в задании 3, со сверкой сходимости и ядра
из задания 7. Без Pinocchio — пропуск.

In [9]:
# Прикладной блок (необязательный): IK на библиотечном якобиане
URDF_2R = ("""<?xml version="1.0"?>
<robot name="planar_2r">
  <link name="link_base"/><link name="link_1"/><link name="link_2"/><link name="link_ee"/>
  <joint name="joint_1" type="revolute">
    <parent link="link_base"/><child link="link_1"/>
    <origin xyz="0 0 0" rpy="0 0 0"/><axis xyz="0 0 1"/>
    <limit lower="-3.14159265358979" upper="3.14159265358979" effort="10" velocity="10"/>
  </joint>
  <joint name="joint_2" type="revolute">
    <parent link="link_1"/><child link="link_2"/>
    <origin xyz="{L1} 0 0" rpy="0 0 0"/><axis xyz="0 0 1"/>
    <limit lower="-3.14159265358979" upper="3.14159265358979" effort="10" velocity="10"/>
  </joint>
  <joint name="joint_ee" type="fixed">
    <parent link="link_2"/><child link="link_ee"/>
    <origin xyz="{L2} 0 0" rpy="0 0 0"/>
  </joint>
</robot>""").format(L1=L1, L2=L2)

try:
    import pinocchio as pin
except ImportError:
    pin = None

if pin is None:
    print('Pinocchio не установлен — блок пропущен.')
    print('Установка: pip install pin (или docker-окружение курса, docker/README.md).')
else:
    model = pin.buildModelFromXML(URDF_2R)
    data = model.createData()
    fid = model.getFrameId('link_ee')

    def fk_pin(q):
        pin.forwardKinematics(model, data, q)
        pin.updateFramePlacements(model, data)
        return data.oMf[fid].translation[:2].copy()

    def jac_pin(q):
        J = pin.computeFrameJacobian(model, data, q, fid, pin.LOCAL_WORLD_ALIGNED)
        return J[:2, :]

    # 1) библиотечный якобиан совпадает с jac2
    errJ = 0.0
    for q_ in rng.uniform(-np.pi, np.pi, (100, 2)):
        errJ = max(errJ, np.abs(jac_pin(q_) - jac2(q_)).max())
    print('100 конфигураций: max || J_pin - jac2 || = %.2e' % errJ)
    assert errJ < 1e-9

    # 2) демпфированный Ньютон (как в Trac-IK/KDL) на той же цели, что задание 3
    lam = 1e-2
    q_pin = np.array([0.0, 1.2])
    hist_pin = [np.linalg.norm(fk_pin(q_pin) - target)]
    for _ in range(60):
        r = fk_pin(q_pin) - target
        J = jac_pin(q_pin)
        dq = -J.T @ np.linalg.solve(J @ J.T + lam ** 2 * np.eye(2), r)
        q_pin = q_pin + dq
        hist_pin.append(np.linalg.norm(fk_pin(q_pin) - target))
        if hist_pin[-1] < 1e-12:
            break
    _, hist_lab = ik2_newton(target, [0.0, 1.2])
    print('та же цель [%.2f, %.2f]: ваш Ньютон — %d итераций,'
          % (target[0], target[1], len(hist_lab) - 1))
    print('библиотечная версия с lambda=%.0e — %d итераций до %.1e'
          % (lam, len(hist_pin) - 1, hist_pin[-1]))
    assert hist_pin[-1] < 1e-9
    print('демпфирование — стандарт промышленных IK-солверов: чуть медленнее,')
    print('но устойчиво у границы рабочей зоны, как показано в задании 5')

    # 3) избыточность: ядро библиотечного якобиана 3R == ядро из задания 7
    URDF_3R = ("""<?xml version="1.0"?><robot name="p3"><link name="b"/><link name="l1"/><link name="l2"/><link name="l3"/><link name="e"/><joint name="j1" type="revolute"><parent link="b"/><child link="l1"/><origin xyz="0 0 0"/><axis xyz="0 0 1"/><limit lower="-3.14159265358979" upper="3.14159265358979" effort="1" velocity="1"/></joint><joint name="j2" type="revolute"><parent link="l1"/><child link="l2"/><origin xyz="{a} 0 0"/><axis xyz="0 0 1"/><limit lower="-3.14159265358979" upper="3.14159265358979" effort="1" velocity="1"/></joint><joint name="j3" type="revolute"><parent link="l2"/><child link="l3"/><origin xyz="{b} 0 0"/><axis xyz="0 0 1"/><limit lower="-3.14159265358979" upper="3.14159265358979" effort="1" velocity="1"/></joint><joint name="je" type="fixed"><parent link="l3"/><child link="e"/><origin xyz="{c} 0 0"/></joint></robot>""").format(a=L[0], b=L[1], c=L[2])
    model3 = pin.buildModelFromXML(URDF_3R)
    data3 = model3.createData()
    J3_pin = pin.computeFrameJacobian(
        model3, data3, q, model3.getFrameId('e'), pin.LOCAL_WORLD_ALIGNED)[:2, :]
    ns_pin = np.linalg.svd(J3_pin)[2][-1]
    ns_lab = np.linalg.svd(J3)[2][-1]
    err_ns = min(np.linalg.norm(ns_pin - ns_lab), np.linalg.norm(ns_pin + ns_lab))
    print('3R: ядро J_pin и ядро J3 из задания 7 — один вектор, расхождение = %.2e'
          % err_ns)
    assert err_ns < 1e-12
    print()
    print('ПРИКЛАДНОЙ БЛОК: ВСЕ ПРОВЕРКИ ПРОЙДЕНЫ')

100 конфигураций: max || J_pin - jac2 || = 1.67e-16
та же цель [0.55, 0.30]: ваш Ньютон — 5 итераций,
библиотечная версия с lambda=1e-02 — 5 итераций до 3.6e-13
демпфирование — стандарт промышленных IK-солверов: чуть медленнее,
но устойчиво у границы рабочей зоны, как показано в задании 5
3R: ядро J_pin и ядро J3 из задания 7 — один вектор, расхождение = 1.78e-16

ПРИКЛАДНОЙ БЛОК: ВСЕ ПРОВЕРКИ ПРОЙДЕНЫ
